# COCO digital channel — resumable

29 reviewed decoders. Use the completed production embedding bank; no new embedding.

Each method: save attacks in temporary ZIPs → decode saved PNGs → save results on Drive → remove attack ZIPs. Start with five images. No Base64 code.

Use **Linux + NVIDIA**, preferably the same A100 runtime as the reviewed bank; high RAM. Download the matching source ZIP too. No GPU runtime is guaranteed to last overnight.


In [ ]:
from pathlib import Path
import os, sys, json, hashlib, shutil, subprocess, zipfile

N_IMAGES = 5                  # After review: change ONLY this to 100; keep the same output.
MODELS = "all"                # All 29; or a list of exact method IDs for diagnostics.
MAX_HOURS = 16                # A session budget, NOT a guarantee of Colab uptime.
SSD_RESERVE_GIB = 60          # Leave SSD room for one isolated decoder environment.
PAIRWISE_TABLES = True
SOURCE_ARCHIVE = Path("COCO_Digital_Channel_RESUMABLE_source.zip")

try:
    from google.colab import drive
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
if os.name != "posix":
    raise RuntimeError("Full 29-method execution requires Linux/NVIDIA: use Colab or a GPU server.")
if IN_COLAB:
    drive.mount("/content/drive")
BASE = Path("/content/drive/MyDrive") if IN_COLAB else Path("/workspace/data")
BANK_ROOT = BASE / "watermarking_production_bank_v1"       # READ ONLY: completed embedding bank.
OUTPUT_ROOT = BASE / "watermarking_digital_channel_v1"    # Durable measurements and reports.
SCRATCH_ROOT = Path("/content/wmbench_digital_scratch") if IN_COLAB else Path("/workspace/wmbench_digital_scratch")
SETTINGS = dict(n_images=N_IMAGES, models=MODELS, max_hours=MAX_HOURS, device="cuda",
    bank=str(BANK_ROOT), output=str(OUTPUT_ROOT), scratch=str(SCRATCH_ROOT),
    ssd_reserve_gib=SSD_RESERVE_GIB, pairwise=PAIRWISE_TABLES, download_author_assets=True,
    manual_assets_root=str(BANK_ROOT / "_manual_assets"))
print("Bank (read-only):", BANK_ROOT)
print("Results:", OUTPUT_ROOT)
print("Temporary fallback:", OUTPUT_ROOT / "_temporary")
print("Per method: save 78 attacked PNGs per available carrier in ZIPs → decode → commit → remove ZIPs.")


## 1. Install the orchestrator and verify its source


In [ ]:
# Upload the matching small source ZIP once; a copy is kept on Drive.
EXPECTED_SHA = "fbaec01ed86f1b0293250e9cde03aca8d1a9a63a0c0a2c69b761b0d1d74577ff"
SAVED_ZIP = OUTPUT_ROOT / "_release" / SOURCE_ARCHIVE.name
if not SOURCE_ARCHIVE.is_file() and SAVED_ZIP.is_file():
    SOURCE_ARCHIVE = SAVED_ZIP
if not SOURCE_ARCHIVE.is_file():
    if not IN_COLAB:
        raise FileNotFoundError("Place the matching source ZIP beside this notebook or set SOURCE_ARCHIVE.")
    from google.colab import files
    uploaded = files.upload()
    matches = [Path(name) for name in uploaded if hashlib.sha256(uploaded[name]).hexdigest() == EXPECTED_SHA]
    if len(matches) != 1:
        raise ValueError("Upload exactly the source ZIP supplied with this notebook.")
    SOURCE_ARCHIVE = matches[0]
data = SOURCE_ARCHIVE.read_bytes()
if hashlib.sha256(data).hexdigest() != EXPECTED_SHA:
    raise ValueError("Source ZIP checksum mismatch. Do not use a different notebook's ZIP.")
SAVED_ZIP.parent.mkdir(parents=True, exist_ok=True)
if not SAVED_ZIP.exists():
    SAVED_ZIP.write_bytes(data)
if hashlib.sha256(SAVED_ZIP.read_bytes()).hexdigest() != EXPECTED_SHA:
    raise ValueError("Saved release readback failed")
PROJECT = SCRATCH_ROOT / "release"
PROJECT.mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(SOURCE_ARCHIVE) as archive:
    for member in archive.infolist():
        target = (PROJECT / member.filename).resolve()
        if not target.is_relative_to(PROJECT.resolve()) or member.is_dir():
            raise ValueError("Unsafe or unexpected source ZIP entry")
        content = archive.read(member)
        if target.exists() and target.read_bytes() != content:
            raise ValueError(f"Source changed: {target}. Use a different empty SCRATCH_ROOT; retain OUTPUT_ROOT.")
        if not target.exists():
            target.parent.mkdir(parents=True, exist_ok=True)
            target.write_bytes(content)

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "uv==0.8.15"], check=True)
import uv
UV = str(uv.find_uv_bin())
controller_dir = SCRATCH_ROOT / "controller"
CONTROLLER = controller_dir / "bin/python"
if not CONTROLLER.exists():
    subprocess.run([UV, "venv", "--python", "3.10.12", str(controller_dir)], check=True)
# This is the CPU attack/report + quality-metric environment, NOT a model environment.
subprocess.run([UV, "pip", "install", "--python", str(CONTROLLER),
    "numpy==1.26.4", "Pillow==10.4.0", "scipy==1.11.4", "scikit-image==0.22.0",
    "psutil==6.1.1", "gdown==5.2.0", "torch==2.5.1", "torchvision==0.20.1", "lpips==0.1.4"], check=True)
ENV = os.environ.copy()
ENV.update(PYTHONPATH=os.pathsep.join([str(PROJECT / "src"), str(PROJECT / "production/src")]),
    PYTHONUNBUFFERED="1", OMP_NUM_THREADS="4", OPENBLAS_NUM_THREADS="1", MKL_NUM_THREADS="4",
    PATH=str(Path(UV).parent)+os.pathsep+ENV.get("PATH", ""),
    UV_PYTHON_INSTALL_DIR=str(SCRATCH_ROOT / "python"))
CONFIG = SCRATCH_ROOT / "digital-settings.json"
CONFIG.write_text(json.dumps(SETTINGS, indent=2), encoding="utf-8")
print("Free SSD: %.1f GiB" % (shutil.disk_usage(SCRATCH_ROOT).free / 1024**3))
subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"], check=True)


## 2. Restore the reviewed system dependencies

Only missing MagickWand/GL libraries are installed; no method is changed.


In [ ]:
# Only RoSteALS/SepWater need these author-import system libraries.
# Restore the setup used by RoSteALS_AUTHOR_FIX and RoSteALS_SepWater_RECOVERY.
# No model source, checkpoint, Python lock, payload or matching rule is changed.
import ctypes
import ctypes.util

def ensure_author_system_dependencies(models, in_colab):
    affected = {"rosteals", "sepwater"}
    selected = affected if models == "all" else affected.intersection(models)
    if not selected:
        return {}

    requirements = {
        "libmagickwand-dev": (
            ("MagickWand-6.Q16", "MagickWand-6.Q16HDRI", "MagickWand-7.Q16HDRI",
             "MagickWand-7.Q16", "MagickWand"), "MagickWandGenesis"),
        "libgl1": (("GL",), None),
    }

    def inspect_libraries():
        found = {}
        for package, (names, symbol) in requirements.items():
            for name in names:
                path = ctypes.util.find_library(name)
                if not path:
                    continue
                try:
                    library = ctypes.CDLL(path)
                    if symbol:
                        getattr(library, symbol)
                except (OSError, AttributeError):
                    continue
                found[package] = path
                break
        return found

    found = inspect_libraries()
    missing = [package for package in requirements if package not in found]
    if missing:
        install = ["apt-get", "-qq", "install", "-y", "--no-install-recommends", *missing]
        if not in_colab or not shutil.which("apt-get") or os.geteuid() != 0:
            raise RuntimeError(
                "RoSteALS/SepWater system libraries are missing. On Debian/Ubuntu run: "
                "sudo apt-get update && sudo " + " ".join(install)
                + ". Then rerun this setup cell; keep the existing output directory.")
        print("Installing missing system libraries for " + ", ".join(sorted(selected))
              + ": " + ", ".join(missing), flush=True)
        subprocess.run(["apt-get", "-qq", "update"], check=True, timeout=600)
        subprocess.run(install, check=True, timeout=600)
        found = inspect_libraries()
        missing = [package for package in requirements if package not in found]
        if missing:
            raise RuntimeError("System-library preflight failed after installation: "
                               + ", ".join(missing) + ". No model has been started.")
    print("RoSteALS/SepWater system-library preflight passed:", found, flush=True)
    return found

AUTHOR_SYSTEM_LIBRARIES = ensure_author_system_dependencies(MODELS, IN_COLAB)


In [ ]:
subprocess.run([str(CONTROLLER), "-m", "wmbench_digital.runner", "validate", "--settings", str(CONFIG)], env=ENV, check=True)


## 3. Run

Keep the same output after disconnects. Valid results are never retried to improve BER. If one method fails technically, the run stops and retains its temporary ZIPs. Logs and completed rows are on Drive.


## StegaStamp Clean replay repair

Only the StegaStamp Clean comparison is changed. No decoder, weights, payloads,
PSNR matching, attacks or other methods are changed. The original source ZIP
and run identity stay the same; completed cases are reused.

For StegaStamp Matched Quality only, a **single changed bit out of 100** may be
accepted when both extractions are non-exact and two additional decodes match
the **first fresh decode exactly**. No voting, best-BER selection or bit repair.
All other Clean mismatches still stop execution. A partial case cannot switch
its Clean reference. Model, dependency and PNG checksum checks remain enabled.

The old bank result is retained. Attack deltas use the actual fresh Clean result.
The discrepancy and all verification decodes are saved under
`watermarking_digital_channel_v1/stegastamp/clean-replay/` and referenced in
the case record. This is an explicit replay-policy exception, not a claim that
the historical and current outputs are identical.

**Run all cells with the same bank/output paths and N_IMAGES=5.** No new ZIP is
needed if the original source ZIP was saved on Drive. If requested, upload the
same `COCO_Digital_Channel_RESUMABLE_source.zip` as before. Do not delete results.


In [ ]:
# Narrow StegaStamp repair; the verified original release is NOT edited.
HOTFIX_SOURCE = r'''
"""Scoped Clean replay handling for the frozen digital-channel release.

No decoder, weight, attack, saved carrier or historical measurement is changed.
Only StegaStamp Matched Quality may accept a stable, single-bit replay drift.
The FIRST fresh extraction is the paired Clean control, never the best retry.
"""

import ast
import hashlib
from pathlib import Path
import sys
import time

VERSION = "stegastamp-clean-replay-v1"
METHOD = "stegastamp"
WORKER_SHA = "57f68d4e9c6134c1f5264703e6e00ac1a6ef55a0040265f5b0c6dfd46ef6cf2e"
FIELDS = ("status", "decoded_bits", "ber", "exact")
REPEATS = 2  # Verification only; neither voting nor selection by BER.


def same_result(left, right):
    return all(left.get(key) == right.get(key) for key in FIELDS)


def drift_positions(case, fresh):
    """A deliberately bounded exception, not a general BER tolerance."""
    bank = case["clean"]
    if case["method"] != METHOD or case["mode"] not in {"matched_39", "matched_42", "matched_45"}:
        return None
    capacity = case["assignment"]["capacity"]
    if capacity["information_bits"] != 100 or capacity["information_indices"] != list(range(100)):
        return None
    expected = case["assignment"]["payload"]
    if len(expected) != 100 or any(bit not in (0, 1) for bit in expected):
        return None
    for row in (bank, fresh):
        bits = row.get("decoded_bits")
        if (row.get("status") != "measured" or row.get("exact") is not False
                or not isinstance(bits, list) or len(bits) != 100
                or any(type(bit) is not int or bit not in (0, 1) for bit in bits)
                or row.get("information_bits") != 100):
            return None
        errors = sum(x != y for x, y in zip(bits, expected))
        if row.get("ber") != errors / 100 or row.get("information_errors") != errors:
            return None
        if row.get("details", {}).get("implementation") != "wibe_onnx":
            return None
    changed = [i for i, (a, b) in enumerate(zip(bank["decoded_bits"], fresh["decoded_bits"])) if a != b]
    return changed if len(changed) == 1 else None


def check_clean(clean, case, adapter, metadata, rng_seed, root, execution, prior, job):
    """Keep original strict checks except the audited StegaStamp replay case."""
    from wmbench.io import read_json, write_json, sha256
    from wmbench_full_payload.storage import EvidenceError, inside
    from wmbench_digital import worker

    root = Path(root)
    name = case["method"]
    historical = case["clean"]
    # A partially decoded carrier must never combine two different Clean controls.
    previous_clean = prior.get("clean")
    if name == METHOD and previous_clean and not same_result(previous_clean, clean):
        write_json(root / name / "clean-replay" / f"resume-conflict-{case['id']}-{execution['session']}.json",
                   {"fix": VERSION, "previous_clean": previous_clean, "fresh": clean,
                    "execution": execution, "accepted": False})
        raise EvidenceError("StegaStamp Clean changed within a partial case; existing attack rows retained")
    if same_result(clean, historical):
        return

    # Preserve the same diagnostic and hard-stop behavior for every other failure.
    diagnostic_name = (case["id"] + "-" + execution["session"] if name == METHOD else case["id"]) + ".json"
    write_json(root / name / "clean-drift" / diagnostic_name,
               {"fresh": clean, "bank": historical, "execution": execution})
    changed = drift_positions(case, clean)
    if changed is None:
        raise EvidenceError("Fresh Clean differs from saved bank control; no attack scores accepted")

    carrier = inside(Path(job["bank"]), case["output"])
    if sha256(carrier) != case["carrier_sha256"]:
        raise EvidenceError("StegaStamp carrier checksum changed during Clean replay")
    repeats = []
    for _ in range(REPEATS):
        if time.time() >= job["deadline"]:
            raise TimeoutError("Session budget reached during StegaStamp Clean verification")
        repeats.append(worker.measured(adapter, carrier, metadata, case, rng_seed))
    stable = all(same_result(clean, row) for row in repeats)
    relative = Path(name) / "clean-replay" / f"{case['id']}-{execution['session']}.json"
    receipt = {"fix": VERSION, "case_key": case["key"], "carrier_sha256": case["carrier_sha256"],
               "bank": historical, "first_fresh": dict(clean), "verification_repeats": repeats,
               "changed_indices_zero_based": changed, "execution": execution, "accepted": stable,
               "rule": "matched-only; exactly one bit; both non-exact; two identical verification extractions",
               "paired_control": "first_fresh", "decoder_changed": False, "bank_changed": False}
    write_json(root / relative, receipt)
    if read_json(root / relative) != receipt:
        raise EvidenceError("StegaStamp replay receipt did not persist")
    if not stable:
        raise EvidenceError("StegaStamp Clean is unstable on repeat; no attack scores accepted")
    clean["clean_replay"] = {"fix": VERSION, "receipt": relative.as_posix(),
                            "bank_ber": historical["ber"], "changed_indices_zero_based": changed,
                            "paired_control": "first_fresh"}
    print(f"STEGASTAMP CLEAN REPLAY {case['id']}: one bit differs; "
          f"bank BER={historical['ber']:.2%}, first fresh BER={clean['ber']:.2%}; "
          f"2/2 repeats stable. Historical bank retained. Audit: {root / relative}", flush=True)


def install_worker_patch():
    """Replace one comparison block in memory; frozen files remain byte-identical."""
    from wmbench_digital import worker

    source = Path(worker.__file__).read_bytes()
    if hashlib.sha256(source).hexdigest() != WORKER_SHA:
        raise RuntimeError("Unexpected worker version; refusing to apply the StegaStamp-only fix")
    tree = ast.parse(source.decode("utf-8"))
    function = next(n for n in tree.body if isinstance(n, ast.FunctionDef) and n.name == "decode")
    matches = 0

    class ReplaceGate(ast.NodeTransformer):
        def visit_For(self, node):
            nonlocal matches
            if (isinstance(node.target, ast.Name) and node.target.id == "field"
                    and isinstance(node.iter, ast.Tuple)
                    and [getattr(e, "value", None) for e in node.iter.elts] == list(FIELDS)):
                matches += 1
                replacement = ast.parse("_check_stegastamp_clean(clean, case, adapter, metadata, "
                                        "rng_seed, root, execution, prior, job)").body[0]
                return ast.copy_location(replacement, node)
            return self.generic_visit(node)

    function = ReplaceGate().visit(function)
    if matches != 1:
        raise RuntimeError("Expected exactly one Clean comparison block")
    module = ast.fix_missing_locations(ast.Module(body=[function], type_ignores=[]))
    worker.__dict__["_check_stegastamp_clean"] = check_clean
    exec(compile(module, str(worker.__file__) + " [StegaStamp replay fix]", "exec"), worker.__dict__)


def redirected_command(command, env):
    """Only the isolated StegaStamp decode child uses this repair entry point."""
    import os
    from wmbench.io import read_json

    command = list(command)
    marker = ["-m", "wmbench_digital.runner", "decode", "--settings"]
    for i in range(len(command) - len(marker)):
        if command[i:i+len(marker)] != marker:
            continue
        job = read_json(command[i+len(marker)])
        if job["model"]["id"] != METHOD:
            return command, env
        command[i+1] = "stegastamp_clean_hotfix"
        env = dict(env)
        env["PYTHONPATH"] = str(Path(__file__).resolve().parent) + os.pathsep + env.get("PYTHONPATH", "")
        return command, env
    return command, env


def main():
    from wmbench_digital import runner
    from wmbench.io import read_json

    action = sys.argv[1] if len(sys.argv) > 1 else ""
    if action == "decode":
        job = read_json(Path(sys.argv[sys.argv.index("--settings")+1]))
        if job["model"]["id"] != METHOD:
            raise RuntimeError("This decode entry point is only for StegaStamp")
        install_worker_patch()
    elif action == "run":
        from wmbench import colab_canary
        original = colab_canary.monitored_process

        def monitored(command, env, *args, **kwargs):
            command, env = redirected_command(command, env)
            return original(command, env, *args, **kwargs)

        colab_canary.monitored_process = monitored
    else:
        raise RuntimeError("Use the original runner for validate/report; repair supports run/decode only")
    runner.main()


if __name__ == "__main__":
    main()
'''
HOTFIX_SOURCE = HOTFIX_SOURCE.lstrip("\n")
HOTFIX_SHA = "60fc74e04651d01346029cb5df21cad61cf1e889069d71d86eed396fc6317842"
if hashlib.sha256(HOTFIX_SOURCE.encode("utf-8")).hexdigest() != HOTFIX_SHA:
    raise ValueError("StegaStamp repair source changed")
HOTFIX_DIR = SCRATCH_ROOT / "repairs/stegastamp-clean-replay-v1"
HOTFIX_DIR.mkdir(parents=True, exist_ok=True)
for folder in (HOTFIX_DIR, OUTPUT_ROOT / "_release/repairs/stegastamp-clean-replay-v1"):
    folder.mkdir(parents=True, exist_ok=True)
    target = folder / "stegastamp_clean_hotfix.py"
    if target.exists() and target.read_bytes() != HOTFIX_SOURCE.encode("utf-8"):
        raise ValueError(f"Different repair already exists: {target}")
    if not target.exists():
        target.write_bytes(HOTFIX_SOURCE.encode("utf-8"))
    if hashlib.sha256(target.read_bytes()).hexdigest() != HOTFIX_SHA:
        raise ValueError("Repair readback failed")
ENV["PYTHONPATH"] = str(HOTFIX_DIR) + os.pathsep + ENV["PYTHONPATH"]
print("StegaStamp-only Clean replay repair enabled; bank and completed results retained.")


## Resume after temporary cache loss

This fixes `FileNotFoundError: .../attack-cache/.../owner.json` during cleanup
of a completed method after Colab loses its temporary disk. Saved results are
still checked before cleanup. An absent ZIP is skipped; an existing ZIP still
requires the original ownership, path and checksum checks.

Decoders, attacks, metrics and the previous StegaStamp repair are unchanged.
Keep the same bank/output paths and `N_IMAGES = 5`; run all cells in order.
The original source ZIP is reused from Drive. Do not delete results or cache
metadata manually. This repair does not regenerate finished measurements.


In [ ]:
# Resume repair only: no changes to decoders, attacks, or saved results.
RESUME_FIX_SOURCE = r'''
"""Make temporary ZIP cleanup idempotent after a Colab runtime reset.

The original release and StegaStamp repair remain unchanged. Completed results
are verified first. Owner/checksum checks remain mandatory for every ZIP that
still exists; an absent ZIP requires no deletion and no owner file.
"""

import ast
import hashlib
from pathlib import Path

VERSION = "digital-missing-cache-v1"
STORAGE_SHA = "69f1f1362d337e8b955b8aee2e2da08e3a25d3cc62f6774c3de6c270bd2bf7de"


def install_cleanup_patch():
    """Move exactly one owner check inside the existing-file deletion branch."""
    from wmbench_digital import storage, runner

    source = Path(storage.__file__).read_bytes()
    if hashlib.sha256(source).hexdigest() != STORAGE_SHA:
        raise RuntimeError("Unexpected cleanup version; refusing to apply resume repair")
    tree = ast.parse(source.decode("utf-8"))
    function = next(n for n in tree.body if isinstance(n, ast.FunctionDef) and n.name == "remove_packages")
    loops = [n for n in function.body if isinstance(n, ast.For)
             and isinstance(n.target, ast.Name) and n.target.id == "manifest"]
    if len(loops) != 1:
        raise RuntimeError("Expected exactly one manifest cleanup loop")
    loop = loops[0]
    owner_test = ast.parse('read_json(folder / "owner.json") != {"identity": identity, "method": method}', mode="eval").body
    exists_test = ast.parse("file.exists()", mode="eval").body
    owner = [n for n in loop.body if isinstance(n, ast.If) and ast.dump(n.test) == ast.dump(owner_test)]
    deletion = [n for n in loop.body if isinstance(n, ast.If) and ast.dump(n.test) == ast.dump(exists_test)]
    if len(owner) != 1 or len(deletion) != 1:
        raise RuntimeError("Expected exactly one owner check and one conditional ZIP deletion")
    loop.body.remove(owner[0])
    deletion[0].body.insert(0, owner[0])
    module = ast.fix_missing_locations(ast.Module(body=[function], type_ignores=[]))
    exec(compile(module, str(storage.__file__) + " [missing-cache resume fix]", "exec"), storage.__dict__)
    runner.remove_packages = storage.remove_packages


def main():
    # Keep the previously supplied StegaStamp handling byte-for-byte unchanged.
    import stegastamp_clean_hotfix

    install_cleanup_patch()
    stegastamp_clean_hotfix.main()


if __name__ == "__main__":
    main()
'''
RESUME_FIX_SOURCE = RESUME_FIX_SOURCE.lstrip("\n")
RESUME_FIX_SHA = "94931da30679bd093a6f1790b4ea3058e869d50f9275f381e13e1101d22e0567"
if hashlib.sha256(RESUME_FIX_SOURCE.encode("utf-8")).hexdigest() != RESUME_FIX_SHA:
    raise ValueError("Resume repair source changed")
RESUME_FIX_DIR = SCRATCH_ROOT / "repairs/digital-missing-cache-v1"
for folder in (RESUME_FIX_DIR, OUTPUT_ROOT / "_release/repairs/digital-missing-cache-v1"):
    folder.mkdir(parents=True, exist_ok=True)
    target = folder / "digital_resume_hotfix.py"
    if target.exists() and target.read_bytes() != RESUME_FIX_SOURCE.encode("utf-8"):
        raise ValueError(f"Different resume repair already exists: {target}")
    if not target.exists():
        target.write_bytes(RESUME_FIX_SOURCE.encode("utf-8"))
    if hashlib.sha256(target.read_bytes()).hexdigest() != RESUME_FIX_SHA:
        raise ValueError("Resume repair readback failed")
ENV["PYTHONPATH"] = str(RESUME_FIX_DIR) + os.pathsep + ENV["PYTHONPATH"]
print("Resume repair enabled: already absent temporary ZIPs need no cleanup.")


In [ ]:
# Stop any previous runner for this output folder before starting.
# After a disconnect rerun the same cells with the same bank/output and source ZIP.
command = [str(CONTROLLER), "-m", "digital_resume_hotfix", "run", "--settings", str(CONFIG)]
process = subprocess.Popen(command, env=ENV, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
    text=True, encoding="utf-8", errors="replace", start_new_session=True)
try:
    for line in process.stdout:
        print(line, end="", flush=True)
    if process.wait():
        raise RuntimeError("Runner stopped. See persistent per-method logs; completed records remain safe.")
except BaseException:
    if process.poll() is None:
        import signal
        os.killpg(process.pid, signal.SIGTERM)
        process.wait()
    raise


## 4. Reports

Report rebuilding does not rerun attacks or load a decoder. Partial coverage remains visible.


In [ ]:
subprocess.run([str(CONTROLLER), "-m", "wmbench_digital.runner", "report", "--settings", str(CONFIG)],
    env=ENV, check=True)
from IPython.display import Markdown, display
display(Markdown((OUTPUT_ROOT / "_reports/REPORT.md").read_text(encoding="utf-8")))
print("Aggregate tables:", OUTPUT_ROOT / "_reports")
print("Each method's tables:", OUTPUT_ROOT / "<method>/tables")
print("Detailed bits, errors and attack parameters:", OUTPUT_ROOT / "<method>/records/*.json.gz")
print("Copy OUTPUT_ROOT to your SSD after completion. Temporary attack ZIPs are intentionally not permanent.")
